# 🧪 01 · Criação dos Dados Sintéticos do Workshop

**Comece por aqui!** Este notebook cria, na **sua conta**, todos os dados que vamos usar nos próximos exercícios.

### ▶️ Como usar
1. No topo do notebook, confira os campos **Catálogo** e **Schema**. No **Databricks Free Edition** deixe o padrão: catálogo `workspace`, schema `workshop_negocio`.
2. Conecte ao compute **Serverless** e clique em **Executar tudo** (*Run all*). Leva de 3 a 6 minutos.
3. **Use o mesmo Catálogo e Schema** em todos os outros notebooks do workshop.

> Você não precisa entender o código abaixo — ele só "fabrica" dados fictícios realistas.

### 🏢 A empresa fictícia: **TelcoBR**
Uma operadora de telecom brasileira com planos móveis (Pré, Controle, Pós), Fibra e Combos, vendidos por lojas próprias, revendas, televendas e canais digitais.

### 📚 Tabelas criadas
| Tabela | O que representa | Linhas (aprox.) |
|---|---|---|
| `planos` | Catálogo de planos e preços | 12 |
| `lojas` | Pontos de venda e canais | ~120 |
| `clientes` | Cadastro de clientes (pessoa física) | 50 mil |
| `contratos` | Linhas/assinaturas de cada cliente | ~60 mil |
| `base_mensal` | Foto mensal de cada contrato (ativo no início do mês? cancelou? ativou?) | ~1 milhão |
| `faturamento_mensal` | Faturas mensais por contrato | ~1 milhão |
| `atendimentos` | Contatos no SAC com nota NPS | ~120 mil |
| `vendas_lojas_bruto` | Vendas "sujas", exportadas de um sistema legado (para o exercício de Visual Data Prep) | ~30 mil |
| `metas_lojas` | Metas mensais de vendas por loja | ~2 mil |
| `vendas_lojas_tratadas` | **Gabarito** do exercício de Data Prep (para os dashboards não dependerem dele) | ~2 mil |

### 🎬 Histórias escondidas nos dados (para os participantes "descobrirem")
1. **Reajuste de março/2026**: os planos Pós, Controle, Fibra e Combo tiveram reajuste de **8,9%** em mar/2026 → o churn sobe e "Preço/Reajuste" vira o principal motivo de cancelamento; dúvidas sobre fatura disparam no SAC.
2. **Nordeste** tem churn maior, puxado por problemas de cobertura/sinal.
3. **Revendas** vendem bem, mas os clientes vindos de revenda cancelam muito nos **primeiros 90 dias**.
4. **URA** tem o pior NPS; **App** tem o melhor. Resolver no primeiro contato aumenta muito a nota.
5. **Fibra** cresce mês a mês e é a categoria mais "fiel" junto com os Combos.
6. **Inadimplência** é bem maior para clientes com score de crédito baixo.

Período dos dados: **nov/2024 a set/2026** (24 meses).

In [0]:
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("schema", "workshop_negocio", "Schema")
CATALOGO = dbutils.widgets.get("catalogo").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

if CATALOGO not in [r[0] for r in spark.sql("SHOW CATALOGS").collect()]:
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOGO}`")
    except Exception as e:
        raise Exception(f"O catálogo '{CATALOGO}' não existe e não foi possível criá-lo. No Free Edition use o catálogo 'workspace'.") from e
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.`{SCHEMA}` COMMENT 'Workshop hands-on para times de negócio - TelcoBR (dados sintéticos)'")
spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
print(f"✅ Os dados serão criados em {CATALOGO}.{SCHEMA}")

✅ Os dados serão criados em workspace.workshop_negocio


In [0]:
from pyspark.sql import functions as F, Window

INICIO, FIM = "2024-11-01", "2026-09-01"   # meses de referência (1º dia do mês)
MES_REAJUSTE = "2026-03-01"
REAJUSTE = 0.089

def escolher(df, coluna, semente, opcoes):
    """Escolha ponderada: opcoes = [(valor, peso), ...]. O sorteio é feito UMA vez numa coluna auxiliar
    (rand() dentro de vários WHEN seria avaliado de forma independente em cada ramo e distorceria os pesos)."""
    total, acc, expr = sum(p for _, p in opcoes), 0.0, None
    r = F.col("_sorteio")
    for valor, peso in opcoes:
        acc += peso / total
        expr = F.when(r < acc, F.lit(valor)) if expr is None else expr.when(r < acc, F.lit(valor))
    return df.withColumn("_sorteio", F.rand(semente)).withColumn(coluna, expr.otherwise(F.lit(opcoes[-1][0]))).drop("_sorteio")

def item(lista, r):
    return F.element_at(F.array(*[F.lit(x) for x in lista]), (F.floor(r * len(lista)) + 1).cast("int"))

def salvar(df, nome, comentario):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome)
    spark.sql(f"COMMENT ON TABLE {nome} IS '{comentario}'")
    print(f"✅ {nome}: {spark.table(nome).count():,} linhas")

## 1. Planos e Lojas (dimensões)

In [0]:
planos = spark.createDataFrame([
    (1,  "Pré Turbo",            "Pré-pago", 0.0,    8,   None),
    (2,  "Pré Mega",             "Pré-pago", 0.0,    15,  None),
    (3,  "Controle 20GB",        "Controle", 59.99,  20,  None),
    (4,  "Controle 35GB",        "Controle", 79.99,  35,  None),
    (5,  "Pós 50GB",             "Pós-pago", 119.99, 50,  None),
    (6,  "Pós 100GB",            "Pós-pago", 169.99, 100, None),
    (7,  "Pós Ilimitado Black",  "Pós-pago", 299.99, 300, None),
    (8,  "Fibra 300 Mega",       "Fibra",    99.99,  None, 300),
    (9,  "Fibra 600 Mega",       "Fibra",    129.99, None, 600),
    (10, "Fibra 1 Giga",         "Fibra",    199.99, None, 1000),
    (11, "Combo Família",        "Combo",    289.99, 150, 600),
    (12, "Combo Premium",        "Combo",    429.99, 400, 1000),
], "plano_id INT, nome_plano STRING, categoria STRING, preco_mensal DOUBLE, franquia_gb INT, velocidade_mbps INT")
salvar(planos, "planos", "Catálogo de planos da TelcoBR. preco_mensal = preço de tabela antes do reajuste de mar/2026 (Pré-pago não tem mensalidade: receita vem de recargas).")

In [0]:
cidades = [  # (cidade, uf, região, peso)
    ("São Paulo","SP","Sudeste",14),("Campinas","SP","Sudeste",4),("Ribeirão Preto","SP","Sudeste",2),
    ("Rio de Janeiro","RJ","Sudeste",8),("Niterói","RJ","Sudeste",2),("Belo Horizonte","MG","Sudeste",5),("Uberlândia","MG","Sudeste",2),
    ("Vitória","ES","Sudeste",2),("Curitiba","PR","Sul",4),("Londrina","PR","Sul",2),("Porto Alegre","RS","Sul",4),
    ("Florianópolis","SC","Sul",3),("Joinville","SC","Sul",2),("Salvador","BA","Nordeste",5),("Recife","PE","Nordeste",4),
    ("Fortaleza","CE","Nordeste",4),("São Luís","MA","Nordeste",2),("Natal","RN","Nordeste",2),("Maceió","AL","Nordeste",2),
    ("João Pessoa","PB","Nordeste",2),("Brasília","DF","Centro-Oeste",4),("Goiânia","GO","Centro-Oeste",3),
    ("Campo Grande","MS","Centro-Oeste",1),("Cuiabá","MT","Centro-Oeste",1),("Manaus","AM","Norte",3),("Belém","PA","Norte",3),("Porto Velho","RO","Norte",1),
]
df_cidades = spark.createDataFrame(cidades, "cidade STRING, uf STRING, regiao STRING, peso INT")

# 120 lojas físicas + 2 canais virtuais
lojas_fisicas = spark.range(1, 121).withColumnRenamed("id", "n")
lojas_fisicas = escolher(lojas_fisicas, "tipo_canal", 11, [("Loja Própria", 40), ("Revenda", 55), ("Quiosque", 25)])
lojas_fisicas = escolher(lojas_fisicas, "cidade", 12, [(c[0], c[3]) for c in cidades])
lojas_fisicas = (lojas_fisicas.join(df_cidades.drop("peso"), "cidade")
    .withColumn("loja_id", F.format_string("LJ-%04d", "n"))
    .withColumn("nome_loja", F.concat(F.when(F.col("tipo_canal") == "Revenda", F.lit("Revenda Parceira ")).when(F.col("tipo_canal") == "Quiosque", F.lit("Quiosque TelcoBR ")).otherwise(F.lit("Loja TelcoBR ")), F.col("cidade"), F.lit(" "), F.col("n")))
    .withColumn("data_inauguracao", F.expr("date_add('2015-01-01', cast(rand(13) * 3000 as int))")))
lojas_virtuais = spark.createDataFrame([
    ("LJ-9001", "Canal Digital (App e Site)", "Digital", "Nacional", "BR", "Nacional"),
    ("LJ-9002", "Central de Televendas", "Televendas", "Nacional", "BR", "Nacional"),
], "loja_id STRING, nome_loja STRING, tipo_canal STRING, cidade STRING, uf STRING, regiao STRING").withColumn("data_inauguracao", F.lit("2015-01-01").cast("date"))
lojas = lojas_fisicas.select("loja_id", "nome_loja", "tipo_canal", "cidade", "uf", "regiao", "data_inauguracao").unionByName(lojas_virtuais)
salvar(lojas, "lojas", "Pontos de venda da TelcoBR: lojas próprias, revendas parceiras, quiosques e canais virtuais (Digital e Televendas).")

## 2. Clientes

In [0]:
nomes_f = ["Ana","Maria","Juliana","Fernanda","Camila","Beatriz","Larissa","Patrícia","Aline","Gabriela","Letícia","Mariana","Renata","Vanessa","Bruna","Carla","Daniela","Sandra","Luana","Tatiane"]
nomes_m = ["João","Pedro","Lucas","Gabriel","Rafael","Bruno","Felipe","Gustavo","Carlos","André","Marcelo","Thiago","Rodrigo","Diego","Eduardo","Paulo","Ricardo","Leonardo","Matheus","José"]
sobrenomes = ["Silva","Santos","Oliveira","Souza","Rodrigues","Ferreira","Alves","Pereira","Lima","Gomes","Costa","Ribeiro","Martins","Carvalho","Almeida","Lopes","Soares","Fernandes","Vieira","Barbosa"]

N_CLIENTES = 50_000
clientes = escolher(escolher(spark.range(1, N_CLIENTES + 1).withColumnRenamed("id", "n"),
                             "cidade", 27, [(c[0], c[3]) for c in cidades]),
                    "canal_aquisicao", 32, [("Digital", 30), ("Loja Própria", 28), ("Revenda", 30), ("Televendas", 12)])
clientes = (clientes
    .withColumn("cliente_id", F.format_string("CLI-%06d", "n"))
    .withColumn("genero", F.when(F.rand(21) < 0.52, "Feminino").otherwise("Masculino"))
    .withColumn("nome", F.concat_ws(" ",
        F.when(F.col("genero") == "Feminino", item(nomes_f, F.rand(22))).otherwise(item(nomes_m, F.rand(23))),
        item(sobrenomes, F.rand(24)), item(sobrenomes, F.rand(25))))
    .withColumn("idade", (18 + F.abs(F.randn(26)) * 18).cast("int"))
    .withColumn("idade", F.when(F.col("idade") > 85, 85 - (F.col("n") % 20)).otherwise(F.col("idade")))
    .withColumn("faixa_etaria", F.expr("CASE WHEN idade < 25 THEN '18-24' WHEN idade < 35 THEN '25-34' WHEN idade < 45 THEN '35-44' WHEN idade < 60 THEN '45-59' ELSE '60+' END"))
    .withColumn("score_credito", F.least(F.lit(950), F.greatest(F.lit(300), (620 + F.randn(28) * 140).cast("int"))))
    # ~40% dos clientes entram durante a janela do workshop (crescimento da base)
    .withColumn("data_adesao", F.when(F.rand(29) < 0.40, F.expr("date_add('2024-11-01', cast(rand(30) * 699 as int))"))
                                .otherwise(F.expr("date_add('2016-01-01', cast(rand(31) * 3200 as int))")))
    .withColumn("email", F.concat(F.lower(F.translate(F.split("nome", " ")[0], "ãáâéêíóôõúç", "aaaeeiooouc")), F.lit("."), F.col("n"), F.lit("@email.com")))
)
clientes = (clientes.join(F.broadcast(df_cidades.drop("peso")), "cidade")
    .withColumn("segmento", F.expr("CASE WHEN score_credito >= 760 AND rand(33) < 0.7 THEN 'Premium' WHEN idade < 30 THEN 'Jovem' ELSE 'Massa' END"))
    .select("cliente_id", "nome", "genero", "idade", "faixa_etaria", "email", "cidade", "uf", "regiao", "segmento", "score_credito", "data_adesao", "canal_aquisicao"))
salvar(clientes, "clientes", "Cadastro de clientes pessoa física da TelcoBR. segmento: Premium (alto score), Jovem (<30 anos) ou Massa. Um cliente pode ter mais de um contrato.")

## 3. Contratos (com a mecânica de churn mês a mês)

In [0]:
cli = spark.table("clientes")
# 1º contrato: móvel/combo conforme segmento; ~22% têm um 2º contrato de Fibra
c1 = cli.withColumn("seq", F.lit(1)).withColumn("data_ativacao", F.col("data_adesao")).withColumn(
    "plano_id", F.expr("""CASE
        WHEN segmento = 'Premium' THEN element_at(array(5,6,7,7,11,12,12,6), cast(rand(41)*8 as int)+1)
        WHEN segmento = 'Jovem'   THEN element_at(array(1,2,2,3,3,4,5,8), cast(rand(42)*8 as int)+1)
        ELSE                           element_at(array(1,1,2,3,3,4,4,5,6,8,9,11), cast(rand(43)*12 as int)+1) END"""))
c2 = (cli.where(F.rand(44) < 0.22).withColumn("seq", F.lit(2))
      .withColumn("data_ativacao", F.expr("date_add(data_adesao, cast(rand(45) * greatest(datediff('2026-09-25', data_adesao), 1) as int))"))
      .withColumn("plano_id", F.expr("element_at(array(8,9,9,10), cast(rand(46)*4 as int)+1)")))
contratos0 = c1.unionByName(c2)

# Loja de origem: canal virtual ou loja física da mesma região do cliente
lj = spark.table("lojas")
lj_fis = lj.where("tipo_canal NOT IN ('Digital','Televendas')").withColumn("canal_grp", F.when(F.col("tipo_canal") == "Quiosque", "Loja Própria").otherwise(F.col("tipo_canal")))
lj_fis = lj_fis.withColumn("idx", F.row_number().over(Window.partitionBy("canal_grp", "regiao").orderBy("loja_id"))) \
               .withColumn("qtd", F.count("*").over(Window.partitionBy("canal_grp", "regiao")))
qtd = lj_fis.select(F.col("canal_grp").alias("canal_aquisicao"), "regiao", "qtd").distinct()
contratos0 = (contratos0.join(F.broadcast(qtd), ["canal_aquisicao", "regiao"], "left")
    .withColumn("idx", (F.floor(F.rand(47) * F.coalesce("qtd", F.lit(1))) + 1).cast("int")))
contratos0 = (contratos0.join(F.broadcast(lj_fis.select(F.col("canal_grp").alias("canal_aquisicao"), "regiao", "idx", F.col("loja_id").alias("loja_fis"))),
                              ["canal_aquisicao", "regiao", "idx"], "left")
    .withColumn("loja_id", F.when(F.col("canal_aquisicao") == "Digital", "LJ-9001").when(F.col("canal_aquisicao") == "Televendas", "LJ-9002").otherwise(F.coalesce("loja_fis", F.lit("LJ-9002"))))
    .withColumn("contrato_id", F.format_string("CTR-%06d-%d", F.regexp_extract("cliente_id", r"(\d+)", 1).cast("int"), "seq")))
contratos0 = contratos0.join(F.broadcast(spark.table("planos").select("plano_id", "categoria", "preco_mensal", "franquia_gb")), "plano_id")
# materializa para "congelar" os sorteios aleatórios (serverless não suporta .cache())
contratos0.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("_tmp_contratos0")
contratos0 = spark.table("_tmp_contratos0")

In [0]:
# Explode um registro por mês de vida do contrato dentro da janela e sorteia o cancelamento (hazard mensal)
meses = spark.sql(f"SELECT explode(sequence(DATE'{INICIO}', DATE'{FIM}', INTERVAL 1 MONTH)) AS mes_referencia")
lj_tipo = spark.table("lojas").select("loja_id", F.col("tipo_canal").alias("tipo_canal_loja"))

vida = (contratos0.crossJoin(F.broadcast(meses))
    .where(F.col("mes_referencia") >= F.trunc("data_ativacao", "month"))
    .join(F.broadcast(lj_tipo), "loja_id", "left")
    .withColumn("meses_casa", F.months_between("mes_referencia", F.trunc("data_ativacao", "month")).cast("int"))
    .withColumn("hazard", F.lit(0.014)
        * F.expr("CASE categoria WHEN 'Pré-pago' THEN 1.7 WHEN 'Controle' THEN 1.25 WHEN 'Pós-pago' THEN 0.8 WHEN 'Fibra' THEN 0.6 ELSE 0.45 END")
        * F.when(F.col("regiao") == "Nordeste", 1.45).when(F.col("regiao") == "Norte", 1.2).otherwise(1.0)
        * F.when((F.col("tipo_canal_loja") == "Revenda") & (F.col("meses_casa") < 3), 3.0).otherwise(1.0)
        * F.when((F.col("mes_referencia") >= MES_REAJUSTE) & (F.col("mes_referencia") < "2026-07-01") & F.col("categoria").isin("Pós-pago", "Controle", "Fibra", "Combo"), 1.9).otherwise(1.0)
        * F.when(F.col("score_credito") < 450, 1.3).otherwise(1.0))
    .withColumn("sorteio", F.rand(51)))
vida.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("_tmp_vida_bruta")
vida = spark.table("_tmp_vida_bruta")

cancel = (vida.where(F.col("sorteio") < F.col("hazard"))
    .groupBy("contrato_id").agg(F.min("mes_referencia").alias("mes_cancelamento")))

vida = (vida.join(cancel, "contrato_id", "left")
    .where(F.col("mes_cancelamento").isNull() | (F.col("mes_referencia") <= F.col("mes_cancelamento"))))
vida.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("_tmp_vida")
vida = spark.table("_tmp_vida")

In [0]:
motivo_cancel = F.expr("""CASE
    WHEN mes_cancelamento >= DATE'2026-03-01' AND mes_cancelamento < DATE'2026-07-01' AND categoria IN ('Pós-pago','Controle','Fibra','Combo') AND rand(61) < 0.55 THEN 'Preço/Reajuste'
    WHEN regiao IN ('Nordeste','Norte') AND rand(62) < 0.35 THEN 'Qualidade do sinal/Cobertura'
    ELSE element_at(array('Portabilidade para concorrente','Portabilidade para concorrente','Preço/Reajuste','Qualidade do sinal/Cobertura',
                          'Atendimento insatisfatório','Mudança de endereço','Dificuldade financeira','Lentidão na internet'), cast(rand(63)*8 as int)+1) END""")

dados_cancel = (vida.where("mes_referencia = mes_cancelamento")
    .select("contrato_id", F.expr("date_add(mes_cancelamento, cast(rand(64)*27 as int))").alias("data_cancelamento"), motivo_cancel.alias("motivo_cancelamento")))

contratos = (contratos0.join(dados_cancel, "contrato_id", "left")
    # contratos cancelados antes da janela não existem na base (simplificação)
    .withColumn("status_contrato", F.when(F.col("data_cancelamento").isNotNull(), "Cancelado").otherwise("Ativo"))
    .select("contrato_id", "cliente_id", "plano_id", "loja_id", "data_ativacao", "data_cancelamento", "status_contrato", "motivo_cancelamento"))
contratos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("_tmp_contratos")

## 4. Base mensal e Faturamento

In [0]:
base_mensal = vida.select(
    "mes_referencia", "contrato_id", "cliente_id", "plano_id", "loja_id",
    (F.trunc("data_ativacao", "month") < F.col("mes_referencia")).alias("ativo_inicio_mes"),
    (F.trunc("data_ativacao", "month") == F.col("mes_referencia")).alias("ativado_no_mes"),
    F.coalesce(F.col("mes_cancelamento") == F.col("mes_referencia"), F.lit(False)).alias("cancelado_no_mes"))
salvar(base_mensal, "base_mensal",
       "Foto mensal de cada contrato. Use para CHURN: churn do mês = contratos cancelados_no_mes / contratos ativo_inicio_mes. Uma linha por contrato por mês em que esteve ativo.")

In [0]:
fat = (vida
    .withColumn("fator_reajuste", F.when((F.col("mes_referencia") >= MES_REAJUSTE) & (F.col("categoria") != "Pré-pago"), 1 + REAJUSTE).otherwise(1.0))
    .withColumn("pro_rata", F.when(F.trunc("data_ativacao", "month") == F.col("mes_referencia"), (30 - F.least(F.dayofmonth("data_ativacao"), F.lit(29))) / 30).otherwise(1.0))
    .withColumn("valor_faturado", F.round(F.when(F.col("categoria") == "Pré-pago", 15 + F.floor(F.rand(71) * 4) * 15)   # recargas R$15..60
        .otherwise(F.col("preco_mensal") * F.col("fator_reajuste") * F.col("pro_rata") * (0.97 + F.rand(72) * 0.06)
                   + F.when(F.rand(73) < 0.12, 9.9 + F.rand(74) * 30).otherwise(0)), 2))
    .withColumn("p_atraso", F.when(F.col("categoria") == "Pré-pago", 0.0).otherwise(F.greatest(F.lit(0.02), 0.32 - (F.col("score_credito") - 300) / 650 * 0.30)
                            * F.when(F.col("mes_referencia") >= MES_REAJUSTE, 1.25).otherwise(1.0)))
    .withColumn("dias_atraso", F.when(F.rand(75) < F.col("p_atraso"), (1 + F.pow(F.rand(76), 2) * 119).cast("int")).otherwise(0))
    # meses recentes: atrasos longos ainda não aconteceram
    .withColumn("dias_atraso", F.least("dias_atraso", F.greatest(F.lit(0), F.datediff(F.lit("2026-10-15"), F.date_add("mes_referencia", 10)))))
    .withColumn("status_pagamento", F.when(F.col("dias_atraso") == 0, "Pago em dia").when(F.col("dias_atraso") > 60, "Em aberto").otherwise("Pago com atraso"))
    .withColumn("valor_pago", F.when(F.col("status_pagamento") == "Em aberto", 0.0).otherwise(F.col("valor_faturado")))
    .withColumn("uso_dados_gb", F.round(F.when(F.col("franquia_gb").isNull(), F.lit(None)).otherwise(F.col("franquia_gb") * F.least(F.lit(1.2), F.rand(77) * 1.1 + 0.1)), 1))
    .withColumn("fatura_id", F.concat(F.lit("FAT-"), F.date_format("mes_referencia", "yyyyMM"), F.lit("-"), F.substring("contrato_id", 5, 20))))
faturamento = fat.select("fatura_id", "mes_referencia", "contrato_id", "cliente_id", "plano_id",
                         F.date_add("mes_referencia", 10).alias("data_vencimento"), "valor_faturado", "valor_pago", "dias_atraso", "status_pagamento", "uso_dados_gb")
salvar(faturamento, "faturamento_mensal",
       "Faturas mensais por contrato. Receita = valor_faturado. Pré-pago = soma de recargas. dias_atraso conta dias após o vencimento. Reajuste de 8,9% a partir de mar/2026 (exceto Pré-pago).")

In [0]:
# Status final: Suspenso quando a última fatura tem mais de 30 dias de atraso
ultima = (spark.table("faturamento_mensal").withColumn("rk", F.row_number().over(Window.partitionBy("contrato_id").orderBy(F.desc("mes_referencia"))))
          .where("rk = 1").select("contrato_id", F.when(F.col("dias_atraso") > 30, "Em aberto").otherwise("ok").alias("ult_status")))
contratos_final = (spark.table("_tmp_contratos").join(ultima, "contrato_id", "left")
    .withColumn("status_contrato", F.when((F.col("status_contrato") == "Ativo") & (F.col("ult_status") == "Em aberto"), "Suspenso").otherwise(F.col("status_contrato")))
    .drop("ult_status"))
salvar(contratos_final, "contratos",
       "Contratos (linhas/assinaturas). status_contrato: Ativo, Suspenso (inadimplente) ou Cancelado. Inclui contratos cancelados durante a janela nov/2024-set/2026.")
spark.sql("DROP TABLE IF EXISTS _tmp_contratos")

## 5. Atendimentos (SAC + NPS)

In [0]:
at = (vida
    .withColumn("p_contato", F.lit(0.075)
        + F.when(F.col("mes_referencia") == F.col("mes_cancelamento"), 0.45).otherwise(0)
        + F.when((F.col("mes_referencia") >= MES_REAJUSTE) & (F.col("mes_referencia") < "2026-06-01") & (F.col("categoria") != "Pré-pago"), 0.06).otherwise(0))
    .where(F.rand(81) < F.col("p_contato")))
at = (escolher(at, "canal_atendimento", 82, [("App", 30), ("WhatsApp", 22), ("URA", 24), ("Chat", 10), ("Loja", 14)])
    .withColumn("motivo_contato", F.expr("""CASE
        WHEN mes_referencia = mes_cancelamento AND rand(83) < 0.6 THEN 'Cancelamento'
        WHEN mes_referencia >= DATE'2026-03-01' AND mes_referencia < DATE'2026-06-01' AND categoria <> 'Pré-pago' AND rand(84) < 0.45 THEN 'Dúvida na fatura'
        WHEN regiao IN ('Nordeste','Norte') AND rand(85) < 0.2 THEN 'Sem sinal/Cobertura'
        ELSE element_at(array('Dúvida na fatura','Cobrança indevida','Lentidão na internet','Sem sinal/Cobertura','Troca de plano','Suporte técnico','Portabilidade','Segunda via de fatura'), cast(rand(86)*8 as int)+1) END"""))
    .withColumn("resolvido_primeiro_contato", F.rand(87) < F.expr("CASE canal_atendimento WHEN 'App' THEN 0.78 WHEN 'WhatsApp' THEN 0.72 WHEN 'Chat' THEN 0.66 WHEN 'Loja' THEN 0.70 ELSE 0.48 END"))
    .withColumn("tempo_resolucao_horas", F.round(F.when(F.col("resolvido_primeiro_contato"), 0.2 + F.rand(88) * 2).otherwise(4 + F.pow(F.rand(89), 2) * 92), 1))
    .withColumn("nota_nps", F.least(F.lit(10), F.greatest(F.lit(0), F.round(
        F.expr("CASE canal_atendimento WHEN 'App' THEN 8.6 WHEN 'WhatsApp' THEN 8.1 WHEN 'Chat' THEN 7.6 WHEN 'Loja' THEN 7.9 ELSE 6.2 END")
        + F.when(F.col("resolvido_primeiro_contato"), 0.9).otherwise(-1.8)
        + F.when(F.col("motivo_contato").isin("Cobrança indevida", "Cancelamento"), -1.2).otherwise(0)
        + (F.rand(90) + F.rand(91) + F.rand(92) - 1.5) * 3.2)).cast("int")))
    .withColumn("data_abertura", F.expr("timestampadd(MINUTE, cast(rand(93) * 60 * 24 * 28 as int), cast(mes_referencia as timestamp))"))
    .withColumn("atendimento_id", F.concat(F.lit("ATD-"), F.lpad(F.abs(F.xxhash64("contrato_id", "mes_referencia")).cast("string"), 19, "0"))))
atendimentos = at.select("atendimento_id", "data_abertura", "cliente_id", "contrato_id", "canal_atendimento", "motivo_contato",
                         "resolvido_primeiro_contato", "tempo_resolucao_horas", "nota_nps")
salvar(atendimentos, "atendimentos",
       "Contatos de clientes com o SAC. nota_nps de 0 a 10 (pesquisa pós-atendimento). NPS = % promotores (9-10) menos % detratores (0-6).")

## 6. Vendas "sujas" das lojas (matéria-prima do exercício de Visual Data Prep)
Simulamos uma exportação de sistema legado com **todos os problemas clássicos**: datas em formatos diferentes, código de loja com espaços e minúsculas,
nomes de produto inconsistentes, valores como texto ("R$ 1.234,56"), UF em minúsculo, status com grafias diferentes, linhas duplicadas e campos vazios.

In [0]:
lojas_fis_ids = [r.loja_id for r in spark.table("lojas").where("tipo_canal NOT IN ('Digital','Televendas')").select("loja_id").collect()]
produtos = [("Pós-Pago", 30), ("pos pago", 6), ("POS PAGO", 4), ("Controle", 22), ("CONTROLE ", 5), ("controle", 3),
            ("Pré-Pago", 12), ("pre pago", 3), ("Fibra", 16), ("FIBRA", 4), ("fibra ótica", 3), ("Combo", 8), ("combo", 2)]
vend = escolher(spark.range(1, 29_001).withColumnRenamed("id", "n"), "produto", 103, produtos)
vend = escolher(vend, "status_venda", 119, [("Aprovada", 70), ("APROVADA", 10), ("aprovada", 4), ("Cancelada", 6), ("cancelada", 2), ("Pendente", 5), (None, 3)])
vend = (vend
    .withColumn("data", F.expr("date_add('2025-01-01', cast(rand(101) * 638 as int))"))
    .withColumn("loja", item(lojas_fis_ids, F.rand(102)))
    .withColumn("r_fmt", F.rand(109))
    .withColumn("valor", F.expr("""CASE WHEN lower(produto) LIKE 'p%s%' THEN 119.99 + rand(104)*180 WHEN lower(produto) LIKE 'contr%' THEN 59.99 + rand(105)*20
                                       WHEN lower(produto) LIKE 'pr%' THEN 15 + rand(106)*45 WHEN lower(produto) LIKE 'fib%' THEN 99.99 + rand(107)*100 ELSE 289.99 + rand(108)*140 END"""))
    .withColumn("valor", F.col("valor") * F.when(F.col("data") >= "2026-03-01", 1.089).otherwise(1.0)))
vend = vend.join(F.broadcast(spark.table("lojas").select(F.col("loja_id").alias("loja"), "uf")), "loja")
r = F.col("r_fmt")
vendas_bruto = vend.select(
    F.format_string("V%07d", "n").alias("id_venda"),
    F.when(r < 0.5, F.date_format("data", "yyyy-MM-dd")).when(r < 0.8, F.date_format("data", "dd/MM/yyyy"))
     .otherwise(F.concat(F.date_format("data", "dd-MM-yyyy"), F.lit(" "), F.format_string("%02d:%02d", (F.rand(110) * 12 + 8).cast("int"), (F.rand(111) * 59).cast("int")))).alias("data_venda"),
    F.when(F.rand(112) < 0.15, F.concat(F.lit(" "), F.col("loja"))).when(F.rand(113) < 0.15, F.lower("loja")).otherwise(F.col("loja")).alias("cod_loja"),
    F.col("produto").alias("produto"),
    F.when(F.rand(114) < 0.02, F.lit(None))
     .when(F.rand(115) < 0.45, F.concat(F.lit("R$ "), F.regexp_replace(F.format_number("valor", 2), r"\.", "#")))
     .when(F.rand(116) < 0.5, F.format_string("%.2f", "valor"))
     .otherwise(F.regexp_replace(F.format_string("%.2f", "valor"), r"\.", ",")).alias("valor_venda"),
    item([f"{a} {b}" for a in ["Carlos", "Juliana", "Marcos", "Patrícia", "Rafael", "Simone", "Tiago", "Vera"] for b in ["A.", "M.", "S."]], F.rand(117)).alias("vendedor"),
    F.when(F.rand(118) < 0.3, F.lower("uf")).otherwise(F.col("uf")).alias("uf"),
    "status_venda",
    F.when(F.rand(120) < 0.04, F.lit(None)).otherwise(F.format_string("CLI-%06d", (F.rand(121) * 50000 + 1).cast("int"))).alias("cpf_cliente_mascarado"))
# "R$ 1,234.56" -> "R$ 1.234,56" (formato brasileiro)
vendas_bruto = vendas_bruto.withColumn("valor_venda", F.when(F.col("valor_venda").startswith("R$"),
    F.translate(F.regexp_replace("valor_venda", ",", "."), "#", ",")).otherwise(F.col("valor_venda")))
vendas_bruto.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("_tmp_vendas")
vendas_bruto = spark.table("_tmp_vendas")
# ~3% de linhas duplicadas (o legado exportou duas vezes)
vendas_bruto = vendas_bruto.unionByName(vendas_bruto.sample(0.03, seed=7))
salvar(vendas_bruto.orderBy(F.rand(122)), "vendas_lojas_bruto",
       "Exportação BRUTA do sistema legado de vendas das lojas físicas. Tem problemas de qualidade: datas em vários formatos, valores como texto, nomes de produto inconsistentes, duplicatas e nulos. Precisa de tratamento antes de usar!")

In [0]:
metas = (spark.table("lojas").where("tipo_canal NOT IN ('Digital','Televendas')")
    .crossJoin(spark.sql("SELECT explode(sequence(DATE'2025-01-01', DATE'2026-09-01', INTERVAL 1 MONTH)) AS mes"))
    .withColumn("meta_qtd_vendas", F.round(F.when(F.col("tipo_canal") == "Loja Própria", 11).when(F.col("tipo_canal") == "Revenda", 9.5).otherwise(7.5) * (0.8 + F.rand(131) * 0.35)).cast("int"))
    .withColumn("meta_receita", F.round(F.col("meta_qtd_vendas") * (150 + F.rand(132) * 20) * F.when(F.col("mes") >= "2026-03-01", 1.089).otherwise(1.0), 2))
    .select("loja_id", "mes", "meta_qtd_vendas", "meta_receita"))
salvar(metas, "metas_lojas", "Metas mensais por loja física: quantidade de vendas aprovadas e receita (R$).")

### Gabarito do Data Prep (`vendas_lojas_tratadas`)
É o resultado esperado do notebook **02 · Visual Data Prep**. Fica disponível para que os dashboards funcionem mesmo que alguém não termine o exercício.

In [0]:
%sql
CREATE OR REPLACE TABLE vendas_lojas_tratadas
COMMENT 'GABARITO do exercício de Visual Data Prep: vendas aprovadas das lojas físicas, limpas, deduplicadas, agregadas por loja e mês e comparadas com a meta.'
AS
WITH limpas AS (
  SELECT DISTINCT
    id_venda,
    coalesce(try_to_date(data_venda, 'yyyy-MM-dd'), try_to_date(data_venda, 'dd/MM/yyyy'), try_to_date(substr(data_venda, 1, 10), 'dd-MM-yyyy')) AS data_venda,
    upper(trim(cod_loja)) AS loja_id,
    CASE WHEN lower(produto) LIKE 'p%s%' THEN 'Pós-pago' WHEN lower(produto) LIKE 'contr%' THEN 'Controle'
         WHEN lower(produto) LIKE 'pr%' THEN 'Pré-pago' WHEN lower(produto) LIKE 'fib%' THEN 'Fibra' ELSE 'Combo' END AS categoria,
    CASE WHEN valor_venda LIKE 'R$%' THEN try_cast(replace(replace(replace(valor_venda, 'R$ ', ''), '.', ''), ',', '.') AS DOUBLE)
         ELSE try_cast(replace(valor_venda, ',', '.') AS DOUBLE) END AS valor_venda,
    initcap(trim(status_venda)) AS status_venda
  FROM vendas_lojas_bruto
),
agregadas AS (
  SELECT loja_id, date_trunc('MONTH', data_venda)::date AS mes, count(*) AS qtd_vendas, round(sum(coalesce(valor_venda, 0)), 2) AS receita_vendas
  FROM limpas WHERE status_venda = 'Aprovada' GROUP BY ALL
)
SELECT a.mes, a.loja_id, l.nome_loja, l.tipo_canal, l.cidade, l.uf, l.regiao,
       a.qtd_vendas, a.receita_vendas, m.meta_qtd_vendas, m.meta_receita,
       round(a.receita_vendas / m.meta_receita, 4) AS pct_atingimento_meta,
       a.receita_vendas >= m.meta_receita AS bateu_meta
FROM agregadas a
JOIN lojas l USING (loja_id)
LEFT JOIN metas_lojas m ON m.loja_id = a.loja_id AND m.mes = a.mes

In [0]:
for t in ["_tmp_contratos0", "_tmp_vida_bruta", "_tmp_vida", "_tmp_vendas"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## 7. Descrições de colunas (ajudam o Genie a entender os dados)

In [0]:
comentarios = {
    "clientes": {"cliente_id": "Identificador único do cliente", "segmento": "Segmento de valor: Premium, Jovem ou Massa", "score_credito": "Score de crédito (300 a 950). Abaixo de 500 = alto risco",
                 "data_adesao": "Data em que o cliente entrou na TelcoBR", "canal_aquisicao": "Canal pelo qual o cliente foi adquirido", "regiao": "Região do Brasil (Sudeste, Sul, Nordeste, Centro-Oeste, Norte)"},
    "contratos": {"status_contrato": "Ativo, Suspenso (última fatura com mais de 30 dias de atraso) ou Cancelado", "data_cancelamento": "Data do cancelamento (nula se ativo)",
                  "motivo_cancelamento": "Motivo informado pelo cliente no cancelamento", "loja_id": "Loja/canal que vendeu o contrato"},
    "base_mensal": {"mes_referencia": "Primeiro dia do mês da foto", "ativo_inicio_mes": "TRUE se o contrato já estava ativo no início do mês (denominador do churn)",
                    "cancelado_no_mes": "TRUE se o contrato cancelou neste mês (numerador do churn)", "ativado_no_mes": "TRUE se o contrato foi ativado neste mês (gross adds)"},
    "faturamento_mensal": {"mes_referencia": "Mês de competência da fatura", "valor_faturado": "Valor faturado em R$ (= receita)", "valor_pago": "Valor efetivamente pago em R$",
                           "dias_atraso": "Dias de atraso após o vencimento (0 = pago em dia)", "status_pagamento": "Pago em dia, Pago com atraso ou Em aberto (> 60 dias)",
                           "uso_dados_gb": "Consumo de dados móveis no mês (GB). Nulo para Fibra"},
    "atendimentos": {"canal_atendimento": "Canal do contato: App, WhatsApp, URA (telefone), Chat ou Loja", "motivo_contato": "Motivo do contato",
                     "resolvido_primeiro_contato": "TRUE se resolvido no primeiro contato (FCR)", "tempo_resolucao_horas": "Horas até a resolução",
                     "nota_nps": "Nota de 0 a 10 da pesquisa pós-atendimento (9-10 promotor, 7-8 neutro, 0-6 detrator)"},
    "planos": {"categoria": "Pré-pago, Controle, Pós-pago, Fibra ou Combo", "preco_mensal": "Preço de tabela em R$ antes do reajuste de mar/2026"},
}
for tabela, cols in comentarios.items():
    for col, txt in cols.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {col} COMMENT '{txt}'")
print("✅ Comentários aplicados")

## 8. Checagem final
Se todas as tabelas abaixo aparecerem com linhas, está tudo pronto! ➡️ Siga para o notebook **02 · Visual Data Prep**.

In [0]:
%sql
SELECT 'clientes' AS tabela, count(*) AS linhas FROM clientes UNION ALL
SELECT 'contratos', count(*) FROM contratos UNION ALL
SELECT 'base_mensal', count(*) FROM base_mensal UNION ALL
SELECT 'faturamento_mensal', count(*) FROM faturamento_mensal UNION ALL
SELECT 'atendimentos', count(*) FROM atendimentos UNION ALL
SELECT 'vendas_lojas_bruto', count(*) FROM vendas_lojas_bruto UNION ALL
SELECT 'vendas_lojas_tratadas', count(*) FROM vendas_lojas_tratadas

### 🔎 Conferindo as histórias: churn mensal por categoria (o pico de mar–jun/2026 é o reajuste)

In [0]:
%sql
SELECT b.mes_referencia, p.categoria,
       round(100 * count_if(b.cancelado_no_mes) / count_if(b.ativo_inicio_mes), 2) AS churn_pct
FROM base_mensal b JOIN planos p USING (plano_id)
GROUP BY ALL ORDER BY 1, 2